# 01 Exploratory Data Analysis (EDA) - Phase 1.1 to 1.4

This notebook covers:
1. **Phase 1.1: Data Ingestion & Initial Profiling**
2. **Phase 1.2: Data Quality Checks**
3. **Phase 1.3: Missing Value Analysis**
4. **Phase 1.4: Univariate Analysis**

**Note:** If you encounter `FileNotFoundError` or `ModuleNotFoundError`, please **restart the kernel** (Kernel -> Restart) to ensure the updated `src/` modules are correctly loaded.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew, kurtosis
from datetime import datetime

try:
    import missingno as msno
except ImportError:
    msno = None

# Add project root to path for internal imports
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)
    print(f"Added {project_root} to sys.path")

from src.utils.data_loader import load_raw
from src.utils.logger import logger

print("Environment setup complete.")

## 1.1 Data Ingestion & Initial Profiling

In [ ]:
df = load_raw()
print(f"Data loaded. Shape: {df.shape}")
df.head()

In [ ]:
print("--- Data Info ---")
df.info()

print("\n--- Descriptive Statistics ---")
df.describe(include='all').transpose()

## 1.2 Data Quality Checks

In [ ]:
print("--- Duplicate Rows ---")
duplicates = df.duplicated().sum()
print(f"Duplicate rows found: {duplicates}")
if duplicates > 0:
    df = df.drop_duplicates()
    print("Duplicates dropped.")

In [ ]:
print("--- Time_of_Booking Analysis ---")
# The dataset currently has categorical time buckets.
print(df['Time_of_Booking'].value_counts())
print("Time_of_Booking is already in categorical form.")

In [ ]:
print("--- Cardinality & Unique Values ---")
cat_cols = ['Location_Category', 'Customer_Loyalty_Status', 'Vehicle_Type', 'Time_of_Booking']
for col in cat_cols:
    print(f"\n{col} uniques ({df[col].nunique()}):")
    print(df[col].unique())

In [ ]:
print("--- Domain Constraint Validation ---")
print(f"Negative Fares: {(df['Historical_Cost_of_Ride'] <= 0).sum()}")
print(f"Zero/Negative Durations: {(df['Expected_Ride_Duration'] <= 0).sum()}")
print(f"Ratings outside [1, 5]: {((df['Average_Ratings'] < 1) | (df['Average_Ratings'] > 5)).sum()}")
print(f"Negative Riders/Drivers: {((df['Number_of_Riders'] < 0) | (df['Number_of_Drivers'] < 0)).sum()}")

## 1.3 Missing Value Analysis

In [ ]:
print("--- Missingness Summary ---")
missing_count = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing_count,
    'Percentage (%)': missing_pct
}).sort_values('Missing Count', ascending=False)

print(missing_df[missing_df['Missing Count'] > 0] if missing_count.sum() > 0 else "No missing values found.")
missing_df

In [ ]:
os.makedirs('../visualization/exploration', exist_ok=True)
if msno:
    print("--- Missingness Matrix ---")
    msno.matrix(df)
    plt.savefig('../visualization/exploration/missing_values.png', bbox_inches='tight')
    plt.show()
else:
    print("--- Missingness Heatmap ---")
    plt.figure(figsize=(10, 6))
    sns.heatmap(df.isnull(), cbar=False, cmap='viridis')
    plt.title("Missing Values Heatmap")
    plt.savefig('../visualization/exploration/missing_values.png', bbox_inches='tight')
    plt.show()

## 1.4 Univariate Analysis

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f"Numeric Columns: {num_cols}")
print(f"Categorical Columns: {cat_cols}")

In [ ]:
print("--- Numeric Distributions & Stats ---")
stats_list = []

for col in num_cols:
    # Calculate stats
    skewness = skew(df[col].dropna())
    kurtVal = kurtosis(df[col].dropna())
    stats_list.append({
        'Column': col,
        'Skewness': skewness,
        'Kurtosis': kurtVal
    })
    
    # Plotting
    fig, (ax_box, ax_hist) = plt.subplots(2, sharex=True, gridspec_kw={"height_ratios": (.15, .85)}, figsize=(10, 6))
    sns.boxplot(x=df[col].dropna(), ax=ax_box, color='skyblue')
    sns.histplot(df[col].dropna(), ax=ax_hist, kde=True, color='blue')
    
    ax_box.set(yticks=[], xlabel='')
    ax_hist.set_title(f'Distribution of {col}\n(Skew: {skewness:.2f}, Kurtosis: {kurtVal:.2f})')
    
    plt.savefig(f'../visualization/exploration/univariate_num_{col.lower()}.png', bbox_inches='tight')
    plt.show()
    
pd.DataFrame(stats_list)

In [ ]:
print("--- Categorical Frequencies ---")

for col in cat_cols:
    plt.figure(figsize=(10, 6))
    sns.countplot(data=df, x=col, hue=col, palette='viridis', legend=False)
    plt.title(f'Frequency of {col}')
    plt.xticks(rotation=45)
    plt.savefig(f'../visualization/exploration/univariate_cat_{col.lower()}.png', bbox_inches='tight')
    plt.show()

In [ ]:
print("--- Log-Transform Visualization for Skewed Features ---")

skewed_cols = [s['Column'] for s in stats_list if abs(s['Skewness']) > 1]

if not skewed_cols:
    print("No highly skewed features found (|skew| > 1).")
else:
    for col in skewed_cols:
        plt.figure(figsize=(12, 5))
        
        # Log transform
        log_data = np.log1p(df[col].dropna())
        
        plt.subplot(1, 2, 1)
        sns.histplot(df[col].dropna(), kde=True, color='blue')
        plt.title(f'Original {col}\n(Skew: {skew(df[col].dropna()):.2f})')
        
        plt.subplot(1, 2, 2)
        sns.histplot(log_data, kde=True, color='green')
        plt.title(f'Log-Transformed {col}\n(Skew: {skew(log_data):.2f})')
        
        plt.savefig(f'../visualization/exploration/univariate_log_{col.lower()}.png', bbox_inches='tight')
        plt.show()

## 1.5 Outlier Detection

In [ ]:
print("--- Outlier Detection (IQR & Z-score) ---")

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
outlier_summary = []

os.makedirs('../visualization/exploration/outlier', exist_ok=True)

def detect_outliers_summary(df, col):
    # IQR Method
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    iqr_outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)].shape[0]
    
    # Z-score Method
    mean_val = df[col].mean()
    std_val = df[col].std()
    z_outliers = ((np.abs(df[col] - mean_val) / std_val) > 3).sum()
    
    return {
        'Column': col,
        'IQR Outliers': iqr_outliers,
        'Z-score Outliers': z_outliers,
        'Lower Bound': lower_bound,
        'Upper Bound': upper_bound
    }

for col in numeric_cols:
    outlier_summary.append(detect_outliers_summary(df, col))
    
    # Visualization
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=df[col])
    plt.title(f"Outlier Detection: {col}")
    plt.savefig(f'../visualization/exploration/outlier/boxplot_{col.lower()}.png', bbox_inches='tight')
    plt.show()

outlier_df = pd.DataFrame(outlier_summary)
outlier_df

### Outlier Handling Strategy
1. **Historical_Cost_of_Ride**: Check if extreme values are legitimate high-duration rides. Since it's the target variable, we should be cautious about dropping them without domain confirmation. Scaling or log-transform (already checked) usually helps.
2. **Number_of_Drivers**: Low supply relative to high demand might create outliers in dynamic pricing logic. 
3. **Conclusion**: At this stage, we will keep outliers but note their impact for the feature engineering phase (e.g., using robust scalers or clipping).

## 1.6 Bivariate Analysis

In [ ]:
print("--- Numeric vs Numeric ---")
os.makedirs('../visualization/exploration/bivariate', exist_ok=True)

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
target = 'Historical_Cost_of_Ride'

for col in [c for c in numeric_cols if c != target]:
    plt.figure(figsize=(10, 6))
    sns.scatterplot(data=df, x=col, y=target, alpha=0.5, color='teal')
    plt.title(f"{col} vs {target}")
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.savefig(f'../visualization/exploration/bivariate/scatter_{col.lower()}_vs_cost.png', bbox_inches='tight')
    plt.show()

In [ ]:
print("--- Categorical vs Numeric ---")
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()

for col in categorical_cols:
    plt.figure(figsize=(12, 6))
    sns.boxplot(data=df, x=col, hue=col, y=target, palette='viridis')
    plt.title(f"{col} vs {target}")
    plt.xticks(rotation=45)
    plt.savefig(f'../visualization/exploration/bivariate/boxplot_{col.lower()}_vs_cost.png', bbox_inches='tight')
    plt.show()

In [ ]:
print("--- Categorical vs Categorical ---")
if 'Location_Category' in df.columns and 'Vehicle_Type' in df.columns:
    plt.figure(figsize=(10, 6))
    sns.countplot(data=df, x='Location_Category', hue='Vehicle_Type', palette='magma')
    plt.title("Vehicle Type Distribution across Location Categories")
    plt.legend(title='Vehicle Type', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.savefig('../visualization/exploration/bivariate/count_location_vs_vehicle.png', bbox_inches='tight')
    plt.show()

## 1.5 Refined Outlier Detection

In [ ]:
print("--- Refined Outlier Detection (Annotated Boxplots & Grubbs' Test) ---")
from scipy import stats
import statsmodels.api as sm

os.makedirs('../visualization/exploration/outlier', exist_ok=True)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

def grubbs_test(data):
    n = len(data)
    mean = np.mean(data)
    std = np.std(data)
    relative_values = np.abs(data - mean)
    index = relative_values.argmax()
    g_stat = relative_values.max() / std
    return g_stat, index

for col in numeric_cols:
    # Detect outliers
    Q1, Q3 = df[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    iqr_outliers = df[(df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)].shape[0]
    z_outliers = (np.abs(stats.zscore(df[col])) > 3).sum()
    g_stat, g_idx = grubbs_test(df[col].values)
    
    # Visualization
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=df[col], palette='coolwarm')
    plt.title(f"Outlier Detection: {col}\n(IQR: {iqr_outliers}, Z>3: {z_outliers}, Grubbs: {g_stat:.2f})")
    plt.annotate(f"Count: {iqr_outliers}", (df[col].max(), 0), textcoords="offset points", xytext=(-50,10), ha='center')
    plt.savefig(f'../visualization/exploration/outlier/refined_boxplot_{col.lower()}.png', bbox_inches='tight')
    plt.show()

# Binary indicator for fare outliers
Q1_f, Q3_f = df['Historical_Cost_of_Ride'].quantile([0.25, 0.75])
IQR_f = Q3_f - Q1_f
df['is_outlier_fare'] = ((df['Historical_Cost_of_Ride'] < Q1_f - 1.5*IQR_f) | (df['Historical_Cost_of_Ride'] > Q3_f + 1.5*IQR_f)).astype(int)
print(f"Fare Outliers detected and flagged: {df['is_outlier_fare'].sum()}")

## 1.6 Refined Bivariate Analysis

In [ ]:
print("--- Demand vs Supply (Density & Stats) ---")
os.makedirs('../visualization/exploration/bivariate', exist_ok=True)
corr_p, p_p = stats.pearsonr(df['Number_of_Riders'], df['Number_of_Drivers'])
corr_s, p_s = stats.spearmanr(df['Number_of_Riders'], df['Number_of_Drivers'])

plt.figure(figsize=(10, 8))
sns.kdeplot(data=df, x='Number_of_Riders', y='Number_of_Drivers', fill=True, cmap='Blues', alpha=0.5)
sns.scatterplot(data=df, x='Number_of_Riders', y='Number_of_Drivers', alpha=0.3, color='black', s=10)
plt.title(f"Demand vs Supply\nPearson: {corr_p:.2f} (p={p_p:.3f}), Spearman: {corr_s:.2f} (p={p_s:.3f})")
plt.savefig('../visualization/exploration/bivariate/demand_vs_supply_density.png', bbox_inches='tight')
plt.show()

In [ ]:
print("--- Duration vs Fare (OLS & LOESS) ---")
plt.figure(figsize=(12, 7))
sns.regplot(data=df, x='Expected_Ride_Duration', y='Historical_Cost_of_Ride', 
            scatter_kws={'alpha':0.3, 's':10}, line_kws={'color':'red', 'label':'OLS'})
sns.regplot(data=df, x='Expected_Ride_Duration', y='Historical_Cost_of_Ride', 
            scatter=False, lowess=True, line_kws={'color':'green', 'label':'LOESS'})

# Simple Linear Regression for R2 and RMSE
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
X_f = df[['Expected_Ride_Duration']]
y_f = df['Historical_Cost_of_Ride']
lr = LinearRegression().fit(X_f, y_f)
preds = lr.predict(X_f)
r2 = r2_score(y_f, preds)
rmse = np.sqrt(mean_squared_error(y_f, preds))

plt.title(f"Duration vs Fare (R²={r2:.3f}, RMSE={rmse:.2f})")
plt.legend()
plt.savefig('../visualization/exploration/bivariate/duration_vs_fare_reg.png', bbox_inches='tight')
plt.show()

In [ ]:
print("--- Categorical Analysis: Location, Vehicle, Loyalty ---")
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# Location vs Fare (Violin + Tukey)
plt.figure(figsize=(12, 6))
sns.violinplot(data=df, x='Location_Category', hue='Location_Category', y='Historical_Cost_of_Ride', palette='Pastel1')
plt.title("Location Category vs Historical Cost")
plt.savefig('../visualization/exploration/bivariate/location_vs_cost_violin.png', bbox_inches='tight')
plt.show()
tukey = pairwise_tukeyhsd(df['Historical_Cost_of_Ride'], df['Location_Category'])
print("Tukey HSD - Location Category:\n", tukey.summary())

# Vehicle Type & Loyalty (Mann-Whitney / ANOVA)
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
sns.boxplot(data=df, x='Vehicle_Type', hue='Vehicle_Type', y='Historical_Cost_of_Ride', ax=axes[0], palette='Set2')
axes[0].set_title("Vehicle Type vs Fare")
sns.boxplot(data=df, x='Customer_Loyalty_Status', hue='Customer_Loyalty_Status', y='Historical_Cost_of_Ride', ax=axes[1], palette='Set3')
axes[1].set_title("Loyalty Status vs Fare")
plt.savefig('../visualization/exploration/bivariate/categorical_facet_plots.png', bbox_inches='tight')
plt.show()

In [ ]:
print("--- Past Rides & Ratings vs Fare ---")
df['Past_Rides_Bin'] = pd.cut(df['Number_of_Past_Rides'], bins=5)
plt.figure(figsize=(12, 6))
sns.boxplot(data=df, x='Past_Rides_Bin', hue='Past_Rides_Bin', y='Historical_Cost_of_Ride', palette='YlOrRd')
plt.title("Binned Past Rides vs Fare")
plt.savefig('../visualization/exploration/bivariate/past_rides_binned.png', bbox_inches='tight')
plt.show()

plt.figure(figsize=(10, 8))
plt.hexbin(df['Average_Ratings'], df['Historical_Cost_of_Ride'], gridsize=20, cmap='inferno')
plt.colorbar(label='Count')
plt.title("Average Ratings vs Fare (Hexbin density)")
plt.savefig('../visualization/exploration/bivariate/ratings_vs_fare_hexbin.png', bbox_inches='tight')
plt.show()
tau, p_tau = stats.kendalltau(df['Average_Ratings'], df['Historical_Cost_of_Ride'])
print(f"Kendall's Tau (Ratings vs Fare): τ={tau:.3f}, p={p_tau:.4f}")

In [ ]:
print("--- Demand-Supply Ratio vs Fare ---")
df['Demand_Supply_Ratio'] = df['Number_of_Riders'] / df['Number_of_Drivers']
plt.figure(figsize=(12, 6))
sns.scatterplot(data=df, x='Demand_Supply_Ratio', y='Historical_Cost_of_Ride', alpha=0.3, s=15)

# Binned means
df['Ratio_Bin'] = pd.cut(df['Demand_Supply_Ratio'], bins=10)
binned_stats = df.groupby('Ratio_Bin')['Historical_Cost_of_Ride'].agg(['mean', 'std']).reset_index()
binned_stats['bin_mid'] = binned_stats['Ratio_Bin'].apply(lambda x: x.mid)

plt.errorbar(binned_stats['bin_mid'], binned_stats['mean'], yerr=binned_stats['std'], fmt='ro-', label='Binned Mean ± 1 SD')
plt.title("Demand-Supply Ratio vs Fare (Key Surge Insight)")
plt.legend()
plt.savefig('../visualization/exploration/bivariate/demand_supply_ratio_insight.png', bbox_inches='tight')
plt.show()

## 1.7 Correlation & Multicollinearity

In [ ]:
print("--- Correlation Analysis (Pearson & Spearman) ---")
os.makedirs('../visualization/exploration/correlation', exist_ok=True)

numeric_df = df.select_dtypes(include=[np.number])

# Pearson Correlation
plt.figure(figsize=(12, 10))
pearson_corr = numeric_df.corr(method='pearson')
sns.heatmap(pearson_corr, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title("Pearson Correlation Matrix")
plt.savefig('../visualization/exploration/correlation/pearson_heatmap.png', bbox_inches='tight')
plt.show()

# Spearman Correlation
plt.figure(figsize=(12, 10))
spearman_corr = numeric_df.corr(method='spearman')
sns.heatmap(spearman_corr, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title("Spearman Rank Correlation Matrix")
plt.savefig('../visualization/exploration/correlation/spearman_heatmap.png', bbox_inches='tight')
plt.show()

# Identify High Correlation pairs (|r| > 0.85)
high_corr = pearson_corr.stack().reset_index()
high_corr.columns = ['Feature 1', 'Feature 2', 'Correlation']
high_corr = high_corr[(high_corr['Correlation'].abs() > 0.85) & (high_corr['Feature 1'] != high_corr['Feature 2'])]
print("High Correlation Pairs (|r| > 0.85):")
print(high_corr.sort_values(by='Correlation', ascending=False))

In [ ]:
print("--- Variance Inflation Factor (VIF) ---")
from statsmodels.stats.outliers_influence import variance_inflation_factor

def calculate_vif(X):
    vif_data = pd.DataFrame()
    vif_data["feature"] = X.columns
    # Add a constant for VIF calculation if intercept is desired, though often omitted for basic VIF
    vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(len(X.columns))]
    return vif_data

# Use only features (dropping target and potential identifiers if they were numeric)
cols_to_drop = ['Historical_Cost_of_Ride', 'is_outlier_fare', 'hour'] # Adjust based on findings
X_vif = numeric_df.drop(columns=[c for c in cols_to_drop if c in numeric_df.columns])

vif_df = calculate_vif(X_vif.dropna())
print("Variance Inflation Factor (VIF):")
print(vif_df.sort_values(by='VIF', ascending=False))

print("\n--- Interpretation ---")
print("VIF > 10 indicates strong multicollinearity. Features with high VIF should be considered for removal or combined.")